In [ ]:
using Pkg
Pkg.activate("../..")

using GeneralizedPerturbedEquilibrium
using Plots
using Printf
using Statistics
using TOML

eq_config = GeneralizedPerturbedEquilibrium.Equilibrium.EquilibriumConfig(TOML.parsefile("gpec.toml")["Equilibrium"], pwd())
equil = GeneralizedPerturbedEquilibrium.Equilibrium.setup_equilibrium(eq_config)
psi_norm = Vector(equil.profiles.xs)

function finite_local_extrema(values::AbstractVector{<:Real})
    extrema = Int[]
    for i in 2:length(values)-1
        if !isfinite(values[i - 1]) || !isfinite(values[i]) || !isfinite(values[i + 1])
            continue
        end
        if (values[i] >= values[i - 1] && values[i] >= values[i + 1]) ||
           (values[i] <= values[i - 1] && values[i] <= values[i + 1])
            push!(extrema, i)
        end
    end
    return extrema
end

function print_extrema_report(name::String, psi::AbstractVector, values::AbstractVector, extrema::AbstractVector{Int})
    println("\n", name, " extrema")
    if isempty(extrema)
        println("  none")
        return
    end
    println("  index\tpsi_norm\tvalue")
    for i in extrema
        @printf("  %d\t%.8f\t%.12e\n", i, psi[i], values[i])
    end
end

s_profile = fill(NaN, length(psi_norm))
alpha_profile = fill(NaN, length(psi_norm))

for i in eachindex(psi_norm)
    try
        ref = GeneralizedPerturbedEquilibrium.ForceFreeStates.salpha_reference(i, equil)
        s_profile[i] = ref.s_ref
        alpha_profile[i] = ref.alpha_ref
    catch err
        @warn "s-alpha reference failed" i psi=psi_norm[i] exception=(err, catch_backtrace())
    end
end

s_extrema = finite_local_extrema(s_profile)
alpha_extrema = finite_local_extrema(alpha_profile)
print_extrema_report("s", psi_norm, s_profile, s_extrema)
print_extrema_report("alpha", psi_norm, alpha_profile, alpha_extrema)

s_mask = isfinite.(s_profile)
alpha_mask = isfinite.(alpha_profile)

p_s = plot(
    psi_norm[s_mask], s_profile[s_mask];
    xlabel="psi_norm",
    ylabel="s",
    label="s",
    lw=2,
    title="s profile",
    framestyle=:box,
)
scatter!(p_s, psi_norm[s_extrema], s_profile[s_extrema]; label="extrema", marker=:circle, color=:red, ms=5)

p_alpha = plot(
    psi_norm[alpha_mask], alpha_profile[alpha_mask];
    xlabel="psi_norm",
    ylabel="alpha",
    label="alpha",
    lw=2,
    title="alpha profile",
    framestyle=:box,
)
scatter!(p_alpha, psi_norm[alpha_extrema], alpha_profile[alpha_extrema]; label="extrema", marker=:circle, color=:red, ms=5)

display(plot(p_s, p_alpha; layout=(1, 2), size=(1100, 420)))


In [ ]:
locstab_fs = zeros(length(psi_norm), 4)
ctrl = GeneralizedPerturbedEquilibrium.ForceFreeStates.ForceFreeStatesControl(verbose=false)

println("Computing Delta' and Di profiles...")
GeneralizedPerturbedEquilibrium.ForceFreeStates.compute_ballooning_stability!(ctrl, locstab_fs, equil; compute_delta_prime=true)

delta_prime = Vector(locstab_fs[:, 4])
di_profile = fill(NaN, length(psi_norm))
psi_mask = abs.(psi_norm) .> eps(Float64)
di_profile[psi_mask] .= locstab_fs[psi_mask, 1] ./ psi_norm[psi_mask]

delta_extrema = finite_local_extrema(delta_prime)
di_extrema = finite_local_extrema(di_profile)
print_extrema_report("Delta'", psi_norm, delta_prime, delta_extrema)
print_extrema_report("Di", psi_norm, di_profile, di_extrema)

delta_mask = isfinite.(delta_prime)
di_mask = isfinite.(di_profile)

p_delta = plot(
    psi_norm[delta_mask], delta_prime[delta_mask];
    xlabel="psi_norm",
    ylabel="Delta'",
    label="Delta'",
    lw=2,
    title="Delta' profile",
    framestyle=:box,
)
scatter!(p_delta, psi_norm[delta_extrema], delta_prime[delta_extrema]; label="extrema", marker=:diamond, color=:red, ms=5)

p_di = plot(
    psi_norm[di_mask], di_profile[di_mask];
    xlabel="psi_norm",
    ylabel="Di",
    label="Di",
    lw=2,
    title="Di profile",
    framestyle=:box,
)
scatter!(p_di, psi_norm[di_extrema], di_profile[di_extrema]; label="extrema", marker=:diamond, color=:red, ms=5)

display(plot(p_delta, p_di; layout=(1, 2), size=(1100, 420)))


In [ ]:
psi_target = 0.974
psi_idx_scan = argmin(abs.(psi_norm .- psi_target))

println("Running s-alpha scan at index $(psi_idx_scan), psi_norm=$(round(psi_norm[psi_idx_scan], digits=6))")

s_scales = collect(range(-5.0, 5.0; length=31))
alpha_scales = collect(range(-5.0, 5.0; length=31))

scan_result = GeneralizedPerturbedEquilibrium.ForceFreeStates.scan_delta_prime_map(
    psi_idx_scan,
    equil;
    theta_k=0.0,
    s_scales=s_scales,
    alpha_scales=alpha_scales,
)

scan_s_raw = scan_result.s_values
scan_pprime_raw = scan_result.pprime_ref .* scan_result.alpha_scales
s_order = sortperm(scan_s_raw)
pprime_order = sortperm(scan_pprime_raw)
scan_s = scan_s_raw[s_order]
scan_pprime = scan_pprime_raw[pprime_order]
scan_delta = scan_result.delta_prime[s_order, pprime_order]
scan_di = scan_result.di_values[s_order, pprime_order]

function symmetric_clims(z)
    finite_vals = vec(z)[isfinite.(vec(z))]
    maxabs = isempty(finite_vals) ? 1.0 : maximum(abs.(finite_vals))
    maxabs = max(maxabs, eps(Float64))
    return (-maxabs, maxabs)
end

p_scan_delta = contourf(
    scan_pprime, scan_s, scan_delta;
    c=cgrad([:blue, :white, :red]),
    clims=symmetric_clims(scan_delta),
    xlabel="pprime",
    ylabel="s",
    title="pprime-s Delta'",
    colorbar_title="Delta'",
    levels=41,
    framestyle=:box,
)
contour!(p_scan_delta, scan_pprime, scan_s, scan_delta; levels=[0.0], color=:black, linewidth=2, label="Delta' = 0")
scatter!(p_scan_delta, [scan_result.reference.pprime_ref], [scan_result.reference.s_ref]; color=:green, marker=:star5, ms=7, label="equilibrium")

p_scan_di = contourf(
    scan_pprime, scan_s, scan_di;
    c=cgrad([:blue, :white, :red]),
    clims=symmetric_clims(scan_di),
    xlabel="pprime",
    ylabel="s",
    title="pprime-s Di",
    colorbar_title="Di",
    levels=41,
    framestyle=:box,
)
contour!(p_scan_di, scan_pprime, scan_s, scan_di; levels=[0.0], color=:black, linewidth=2, label="Di = 0")
scatter!(p_scan_di, [scan_result.reference.pprime_ref], [scan_result.reference.s_ref]; color=:green, marker=:star5, ms=7, label="equilibrium")

display(plot(p_scan_delta, p_scan_di; layout=(1, 2), size=(1150, 460)))


In [ ]:
p_zero = plot(
    xlabel="pprime",
    ylabel="s",
    title="pprime-s zero contours",
    framestyle=:box,
)
contour!(p_zero, scan_pprime, scan_s, scan_delta;
    levels=[0.0], color=:blue, linewidth=3,
    label="Delta' = 0", colorbar=false)

contour!(p_zero, scan_pprime, scan_s, scan_di;
    levels=[0.0], color=:red, linewidth=3, linestyle=:dash,
    label="Di = 0", colorbar=false)


scatter!(p_zero, [scan_result.reference.pprime_ref], [scan_result.reference.s_ref]; color=:green, marker=:star5, ms=8, label="equilibrium")
display(p_zero)
